## **Introduction**

This notebook explores the **modular implementation of AlexNet’s convolutional blocks** using **PyTorch**. The focus is on creating reusable **CNN components** that simplify deep network construction while maintaining flexibility.  

### **What You Will Learn**
- **Understanding AlexNet Architecture** – Breaking down key components like **convolutional layers, ReLU activations, pooling, and normalization**.
- **Building a Modular Convolutional Block** – Implementing **parameterized CNN blocks** that can be reused in deeper networks.
- **Enhancing Feature Extraction** – Exploring **local response normalization (LRN) and max pooling** to improve feature contrast and dimensionality reduction.
- **Constructing the Full AlexNet Model** – Using modular blocks to assemble the classic **AlexNet architecture** for large-scale image classification.

### **Key Highlights**
- **Reusable CNN blocks** for efficient model design.
- **Configurable normalization and pooling** to improve feature learning.
- **Scalable design** that allows deeper network extensions.

By the end of this notebook, you will have a solid grasp of **modular deep learning design** and how **AlexNet revolutionized computer vision** through structured convolutional layers. 🚀


## **AlexNet Architecture**

https://www.kaggle.com/code/blurredmachine/alexnet-architecture-a-complete-guide


<p align="center">
  <img src="https://www.researchgate.net/publication/343107859/figure/fig3/AS:917413931520000@1595739934204/Diagram-of-AlexNet-the-CNN-architecture-proposed-in-18.png" width="70%">
</p>


## **Module Initialization: Importing Required Libraries**  


In this step, we import the necessary **PyTorch libraries** to define and analyze deep learning models.

### **Imported Libraries**
1. **`torch`** – The core PyTorch library for tensor computations and deep learning operations.
2. **`torch.nn`** – Provides neural network layers (`nn.Module`), activation functions, loss functions, and optimization utilities.
3. **`torchsummary.summary`** – Generates a model summary, displaying layer information, output shapes, and total parameters.

These imports set up the foundation for defining, training, and analyzing deep learning models in **PyTorch**.

In [1]:
# Importing the necessary libraries

import torch
import torch.nn as nn
from torchsummary import summary


## **Defining the AlexNet Convolutional Block**  

This class implements a **modular block for the AlexNet architecture**.  
Each **AlexNetBlock** consists of a **convolutional layer**, **ReLU activation**, and optionally, **local response normalization (LRN)** and **max pooling**.

### **Components of the AlexNet Block**
1. **Convolutional Layer (`nn.Conv2d`)**  
   - Extracts spatial features from input images using learnable filters.  
2. **ReLU Activation (`nn.ReLU()`)**  
   - Introduces non-linearity to improve model expressiveness.  
3. **Local Response Normalization (`nn.LocalResponseNorm`)** *(Optional)*  
   - Mimics biological lateral inhibition to enhance feature contrast.  
4. **Max Pooling (`nn.MaxPool2d`)** *(Optional)*  
   - Reduces spatial dimensions while retaining key features.  

### **Key Features**
- **`pool_and_norm=True` (Default):** Applies **normalization and pooling** after the convolutional layer.  
- **Flexible Design:** Allows skipping pooling and normalization when `pool_and_norm=False`.  
- **Reusable Block:** Can be used to construct deeper networks like **AlexNet**.  

This modular block helps in **building deep CNN architectures efficiently**.


In [2]:

# Define a modular convolutional block for AlexNet
class AlexNetBlock(nn.Module):
    def __init__(
        self,
        in_channels,     # Number of input channels (depth of input feature maps)
        out_channels,    # Number of output channels (depth of output feature maps)
        kernel_size,     # Size of convolution filter
        stride,          # Stride of the convolution
        padding,         # Padding to maintain spatial dimensions
        pool_and_norm: bool = True  # Apply pooling and normalization by default
    ) -> None:

        super(AlexNetBlock, self).__init__()

        # Define the convolutional layer
        self.conv_layer = nn.Conv2d(
            in_channels, out_channels, kernel_size, stride, padding
        )

        # Apply ReLU activation function
        self.relu = nn.ReLU()

        # Enable optional local response normalization and max pooling
        self.pool_and_norm = pool_and_norm
        if pool_and_norm:
            self.norm_layer = nn.LocalResponseNorm(
                size=5, alpha=0.0001, beta=0.75, k=2  # Normalization parameters
            )
            self.pool_layer = nn.MaxPool2d(
                kernel_size=3, stride=2  # Max pooling reduces feature map size
            )

    def forward(self, x):
        """
        Forward pass of the AlexNet block.
        """
        x = self.conv_layer(x)  # Apply convolution
        x = self.relu(x)  # Apply activation function

        # Apply optional normalization and pooling
        if self.pool_and_norm:
            x = self.norm_layer(x)
            x = self.pool_layer(x)

        return x  # Return the processed output


## **Defining the AlexNet Model**  



This class implements the **AlexNet architecture**, a pioneering deep CNN introduced in 2012. AlexNet significantly improved **image classification performance** on the **ImageNet dataset** and introduced key deep learning techniques.

### **AlexNet Architecture Breakdown**
1. **Five Convolutional Blocks (`AlexNetBlock`)**  
   - Each block consists of a **convolutional layer**, **ReLU activation**, and optionally **local response normalization (LRN) and max pooling**.  
   - Early layers learn **basic edge and texture features**, while deeper layers capture **complex patterns**.  

2. **Fully Connected Layers**  
   - After convolutions, feature maps are **flattened** into a 1D vector.  
   - **Two fully connected layers** (FC) with **4096 neurons** each process high-level features.  
   - **ReLU activation** introduces non-linearity.  

3. **Dropout Regularization (`nn.Dropout`)**  
   - **Prevents overfitting** by randomly setting neuron outputs to zero during training.  
   - **Dropout rate = 50%** (`p=0.5`).  

4. **Final Classification Layer**  
   - **Softmax output layer (`nn.Linear`)** predicts **`num_classes`** (default: **1000 classes for ImageNet**).  

### **Key Features**
- **Uses `AlexNetBlock` for modularity and efficiency.**  
- **Includes dropout layers** to improve generalization.  
- **Can classify images into a customizable number of classes (`num_classes`).**  
- **Trained on large datasets like ImageNet but adaptable to other datasets.**  

This structure makes **AlexNet highly effective for image classification** while being computationally efficient.


In [3]:
# Define the AlexNet model
class AlexNet(nn.Module):
    def __init__(self, num_classes=1000, in_channels=3) -> None:
        """
        Initializes the AlexNet architecture.

        Args:
        - num_classes (int): Number of output classes (default: 1000 for ImageNet).
        - in_channels (int): Number of input channels (default: 3 for RGB images).
        """
        super(AlexNet, self).__init__()

        # First convolutional block: 3 input channels → 96 filters (11x11 kernel, stride 4, padding 0)
        self.block1 = AlexNetBlock(in_channels, 96, 11, 4, 0, pool_and_norm=True)

        # Second convolutional block: 96 → 256 filters (5x5 kernel, stride 1, padding 2)
        self.block2 = AlexNetBlock(96, 256, 5, 1, 2, pool_and_norm=True)

        # Third convolutional block: 256 → 384 filters (3x3 kernel, stride 1, padding 1)
        self.block3 = AlexNetBlock(256, 384, 3, 1, 1, pool_and_norm=False)

        # Fourth convolutional block: 384 → 384 filters (3x3 kernel, stride 1, padding 1)
        self.block4 = AlexNetBlock(384, 384, 3, 1, 1, pool_and_norm=False)

        # Fifth convolutional block: 384 → 256 filters (3x3 kernel, stride 1, padding 1)
        self.block5 = AlexNetBlock(384, 256, 3, 1, 1, pool_and_norm=True)

        # Flatten layer to convert feature maps into a 1D vector
        self.flatten = nn.Flatten()

        # Fully connected layer 1: 256*6*6 → 4096 neurons
        # Input: 256 * 6 * 6 = 9,216 neurons. Previous convolution layer outputs feature maps with 256 channels, each 6×6 pixels. Flattened to 9216 neurons
        # Output: 4,096 neurons
        self.fc1 = nn.Linear(256 * 6 * 6, 4096)
        self.dropout1 = nn.Dropout(0.5)  # Dropout for regularization

        # Fully connected layer 2: 4096 → 4096 neurons
        self.fc2 = nn.Linear(4096, 4096)
        self.dropout2 = nn.Dropout(0.5)  # Dropout for regularization

        # Final classification layer: 4096 → num_classes
        self.classification_layer = nn.Linear(4096, num_classes)

    def forward(self, x):
        """
        Defines the forward pass of the AlexNet model.
        """
        # Pass through convolutional blocks
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)

        # Flatten feature maps before fully connected layers
        x = self.flatten(x)

        # Fully connected layers with dropout
        x = self.fc1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.dropout2(x)

        # Final classification layer
        x = self.classification_layer(x)

        return x  # Output logits (softmax will be applied during loss computation)


## **Model Summary: AlexNet Architecture Overview**  



In this step, we **instantiate the AlexNet model** and generate a **layer-wise summary** using `torchsummary.summary()`.  

### **Key Components**
1. **Model Instantiation (`model = AlexNet()`)**  
   - Creates an instance of the `AlexNet` class.  
   - Uses default parameters (`num_classes=1000`, `in_channels=3`).  

2. **Generating Model Summary (`summary()`)**  
   - Displays each layer’s **output shape** and **number of trainable parameters**.  
   - Ensures the network is correctly **configured for input images of size (3, 227, 227)**.  
   - Helps **debug architectural mistakes** before training.  

### **Expected Output**
The summary provides:
- **Layer names** (Convolution, Pooling, Fully Connected).  
- **Output tensor shapes at each stage**.  
- **Total number of parameters (trainable + non-trainable).**  

This ensures that **AlexNet is structured correctly and ready for training.**


In [4]:
# Instantiate the AlexNet model
model = AlexNet()

# Generate a detailed summary of the model architecture
summary(model, input_size=(3, 227, 227))  # Input: RGB image of size 227x227


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 96, 55, 55]          34,944
              ReLU-2           [-1, 96, 55, 55]               0
 LocalResponseNorm-3           [-1, 96, 55, 55]               0
         MaxPool2d-4           [-1, 96, 27, 27]               0
      AlexNetBlock-5           [-1, 96, 27, 27]               0
            Conv2d-6          [-1, 256, 27, 27]         614,656
              ReLU-7          [-1, 256, 27, 27]               0
 LocalResponseNorm-8          [-1, 256, 27, 27]               0
         MaxPool2d-9          [-1, 256, 13, 13]               0
     AlexNetBlock-10          [-1, 256, 13, 13]               0
           Conv2d-11          [-1, 384, 13, 13]         885,120
             ReLU-12          [-1, 384, 13, 13]               0
     AlexNetBlock-13          [-1, 384, 13, 13]               0
           Conv2d-14          [-1, 384,

## **Interpretation of Results**


Once the AlexNet model is trained, we assess its performance using key metrics such as **loss, accuracy, and predictions**.

### **1. Understanding Accuracy and Loss**
- **Training Loss:** A steadily decreasing training loss indicates that the model is effectively learning. However, if validation loss begins to increase while training loss continues to drop, it suggests **overfitting**.
- **Validation Loss:** A low validation loss suggests good generalization, whereas a high validation loss with low training loss indicates that the model is overfitting.
- **Training vs Validation Accuracy:**
  - If training accuracy is significantly higher than validation accuracy, the model may be **memorizing the training data** instead of generalizing to new examples.
  - A **steady validation accuracy increase** over epochs suggests that the model is learning effectively.

### **2. Evaluating Model Predictions**
- **Correct Predictions:** A high percentage of correctly classified images indicates strong feature extraction and learning.
- **Misclassified Images:** Analyzing misclassified images can highlight patterns where the model struggles, such as:
  - Classes that look visually similar being confused.
  - The need for additional data augmentation to improve generalization.
  - Potential dataset biases affecting classification.

## **Conclusion**

In this notebook, we implemented and analyzed the **AlexNet** architecture using PyTorch. Below are the key takeaways:

### **Key Learnings**
- **AlexNet Architecture:** We built a deep convolutional neural network with multiple layers for feature extraction and classification.
- **Training Performance:** The model was trained using an optimization algorithm, and we monitored its learning through loss and accuracy curves.
- **Evaluation:** By analyzing validation accuracy and sample predictions, we gained insights into the model’s performance and areas for improvement.
